In [ ]:
# from http.server import HTTPServer, BaseHTTPRequestHandler

# class Handler(BaseHTTPRequestHandler):
#     def do_GET(self):
#         from urllib.parse import urlparse, parse_qs
#         query = urlparse(self.path).query
#         params = parse_qs(query)
#         self.server.code = params.get("code", [None])[0]
#         self.send_response(200)
#         self.end_headers()
#         self.wfile.write(b"You can close this tab now!")

# httpd = HTTPServer(('localhost', 1717), Handler)
# print("Waiting for redirect with code...")
# httpd.handle_request()  # Waits for **one** request
# auth_code = httpd.code

In [ ]:
import os
import requests
import hashlib
import base64
import secrets
from urllib.parse import urlencode, urlparse, parse_qs
from http.server import HTTPServer, BaseHTTPRequestHandler
from dotenv import load_dotenv

# --- Load environment variables ---
dotenv_path = "/home/simonhans/coding/sf_majick/.env"  # adjust to your path
load_dotenv(dotenv_path=dotenv_path)

CLIENT_ID = os.environ.get("SF_CLIENT_ID")
CLIENT_SECRET = os.environ.get("SF_CLIENT_SECRET")
REDIRECT_URI = "http://localhost:8080/callback"  # port must match HTTPServer below

# --- Step 0: Generate PKCE code_verifier and code_challenge ---
code_verifier = secrets.token_urlsafe(64)
code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode()).digest()
).rstrip(b"=").decode("utf-8")

# --- Step 1: Construct authorization URL with PKCE ---
auth_params = {
    "response_type": "code",
    "client_id": CLIENT_ID,
    "redirect_uri": REDIRECT_URI,
    "code_challenge": code_challenge,
    "code_challenge_method": "S256"
}

auth_url = f"https://login.salesforce.com/services/oauth2/authorize?{urlencode(auth_params)}"
print("Open this URL in your browser and log in:", auth_url)

# --- Step 2: Start temporary HTTP server to capture the code ---
class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        query = urlparse(self.path).query
        params = parse_qs(query)
        self.server.code = params.get("code", [None])[0]
        self.send_response(200)
        self.end_headers()
        self.wfile.write(b"You can close this tab now!")

httpd = HTTPServer(('localhost', 8080), Handler)
print("Waiting for redirect with code on localhost:1717...")
httpd.handle_request()  # waits for a single request
auth_code = httpd.code
print("Authorization code received!")

# --- Step 3: Exchange code for access + refresh token using PKCE ---
token_url = "https://login.salesforce.com/services/oauth2/token"
payload = {
    "grant_type": "authorization_code",
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "redirect_uri": REDIRECT_URI,
    "code": auth_code,
    "code_verifier": code_verifier
}

r = requests.post(token_url, data=payload)
r.raise_for_status()
tokens = r.json()

print("Access Token:", tokens.get("access_token"))
print("Refresh Token:", tokens.get("refresh_token"))
print("Instance URL:", tokens.get("instance_url"))

# --- Step 4: Append refresh token + instance URL to .env ---
with open(".env", "a") as f:
    f.write(f"SF_REFRESH_TOKEN={tokens.get('refresh_token')}\n")
    f.write(f"SF_INSTANCE_URL={tokens.get('instance_url')}\n")

print("Refresh token and instance URL appended to .env")